# Demo Notebook

This notebook demonstrates the capabilities of the fine-tuned **BERT Mountain NER** model. 
The model extracts mountain names (`B-MOUNTAIN`, `I-MOUNTAIN`) from unstructured text.

In [ ]:
!pip install -q transformers datasets ipywidgets

In [2]:
from transformers import pipeline
import ipywidgets as widgets
from IPython.display import display, HTML

Loading model from Hugging Face Hub

In [3]:
MODEL_ID = "TerentievaSof/bert-base-mountain-ner"

ner_pipeline = pipeline(
    "token-classification",
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
def visualize_ner(text: str):
    """Highlight detected mountains directly in HTML text."""

    predictions = ner_pipeline(text)
    if not predictions:
        print("No mountains detected.")
        return
        
    sorted_preds = sorted(predictions, key=lambda x: x['start'], reverse=True)
    
    html_text = text
    for pred in sorted_preds:
        start, end = pred['start'], pred['end']
        entity_text = text[start:end]
        score = pred['score'] * 100
                
        badge = (
            f'<mark style="background-color: #85e3b2; padding: 2px 6px; border-radius: 4px; font-weight: bold;">'
            f'{entity_text} <span style="font-size: 0.75em; color: #333;">({score:.1f}%)</span>'
            f'</mark>'
        )
        html_text = html_text[:start] + badge + html_text[end:]
        
    display(HTML(f"<div style='font-size: 16px; line-height: 1.8;'>{html_text}</div>"))

In [5]:
# 1. Classical / Famous mountains
visualize_ner("Yesterday we reached the summit of Mount Rainier and Mount Everest after a long hike.")

# 2. Local / Transliterated peaks
visualize_ner("We had an amazing trip near Pip Ivan and Hoverla in the Carpathian mountains.")

# 3. Unseen / Fictional names (Generalization test)
visualize_ner("They spent three days climbing Mount Dragonspire last weekend.")

# 4. Negative test (No false positives)
visualize_ner("I decided to buy a cup of hot coffee and go for a walk in Paris.")

No mountains detected.


In [6]:
text_input = widgets.Textarea(
    value="We set up a base camp near Mount McKinley before attempting the climb.",
    placeholder="Type any English sentence here...",
    description="Text:",
    layout=widgets.Layout(width="80%", height="80px")
)

button = widgets.Button(description="Extract Mountains", button_style="success")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        visualize_ner(text_input.value)

button.on_click(on_button_clicked)

ui = widgets.VBox([text_input, button, output])

print("Try your own sentence below:")
display(ui)

Try your own sentence below:
